# 🚗 RoadSense — YOLOv11 Road Anomaly Detection

Training notebook to detect road anomalies using transfer learning.

**Classes:** Pothole · Fallen-Barrier · Fallen-Cone · Fallen-Pole
**Dataset:** RoadFix v3 — 23,876 images (Roboflow)
**Model:** YOLOv11s pretrained on COCO

> ⚠️ Before running: go to **Runtime → Change runtime type → T4 GPU**


## ✅ Step 1 — Check GPU

Confirms a GPU is assigned. If output shows 'No devices found', enable GPU in Runtime settings.

In [ ]:
!nvidia-smi

Mon Jun  8 08:45:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 📦 Step 2 — Install Dependencies

Installs Ultralytics (YOLOv11) and Roboflow SDK.

In [ ]:
!pip install ultralytics roboflow -q
import ultralytics
ultralytics.checks()


Ultralytics 8.4.61 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 47.3/112.6 GB disk)


## 💾 Step 3 — Mount Google Drive

All checkpoints and the final model are saved directly to your Drive. This protects your progress if Colab disconnects.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_PROJECT_PATH = '/content/drive/MyDrive/RoadSense/runs'
os.makedirs(DRIVE_PROJECT_PATH, exist_ok=True)
print(f'✅ Drive mounted. Checkpoints will save to: {DRIVE_PROJECT_PATH}')


Mounted at /content/drive
✅ Drive mounted. Checkpoints will save to: /content/drive/MyDrive/RoadSense/runs


## ⚙️ Step 4 — Configuration

**Fill in your Roboflow private API key.** Everything else is pre-filled for the RoadFix dataset.

Find your private API key at: Roboflow → Settings → API Keys

In [ ]:
# ── Roboflow ────────────────────────────────────────
RF_API_KEY = 'YOUR_ROBOFLOW_API_KEY'  # ← paste your key here
RF_WORKSPACE = 'dequillaprojects'
RF_PROJECT   = 'roadfix'
RF_VERSION   = 3

# ── Training ────────────────────────────────────────
MODEL       = 'yolo11s.pt'   # pretrained weights (auto-downloaded)
EPOCHS      = 100
BATCH       = 16
IMG_SIZE    = 640
SAVE_EVERY  = 10             # save checkpoint every N epochs
PATIENCE    = 20             # early stop if no improvement for N epochs
RUN_NAME    = 'roadfix_yolo11s'

print('✅ Config set.')


✅ Config set.


## 📥 Step 5 — Download Dataset from Roboflow

Downloads the RoadFix dataset (23,876 images, YOLOv11 format) directly into Colab. No manual upload needed.

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key=RF_API_KEY)
project = rf.workspace(RF_WORKSPACE).project(RF_PROJECT)
dataset = project.version(RF_VERSION).download('yolov11')

DATA_YAML = dataset.location + '/data.yaml'
print(f'✅ Dataset ready at: {dataset.location}')
print(f'✅ data.yaml: {DATA_YAML}')


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to roadfix-3 in yolov11:: 100%|██████████| 47757/47757 [00:33<00:00, 1409.46it/s] 


✅ Dataset ready at: /content/roadfix-3
✅ data.yaml: /content/roadfix-3/data.yaml


## 🚀 Step 6 — Train Model (Fresh Start)

Starts a new training run. Checkpoints are saved to Google Drive every `SAVE_EVERY` epochs automatically.

> ⚠️ **Skip this cell** if you are resuming after a crash — use Step 7 or Step 8 instead.

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL)

results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    save_period=SAVE_EVERY,      # saves epoch10.pt, epoch20.pt ... to Drive
    project=DRIVE_PROJECT_PATH,
    name=RUN_NAME,
    exist_ok=True,
    patience=PATIENCE,
    verbose=True
)

print(f'\n✅ Training complete!')
print(f'Best model saved at: {DRIVE_PROJECT_PATH}/{RUN_NAME}/weights/best.pt')


Ultralytics 8.4.61 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/roadfix-3/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=roadfix_yolo11s, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, pati

KeyboardInterrupt: 

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/drive/MyDrive/RoadSense/runs/roadfix_yolo11s/weights/best.pt")
model.export(format="onnx", simplify=True)

Ultralytics 8.4.61 🚀 Python-3.12.13 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLO11s summary (fused): 101 layers, 9,414,348 parameters, 0 gradients, 21.3 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/RoadSense/runs/roadfix_yolo11s/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 8, 8400) (54.3 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 12 packages in 354ms
Prepared 4 packages in 1.71s
Installed 4 packages in 244ms
 + colorama==0.4.6
 + onnx==1.21.0
 + onnxruntime==1.26.0
 + onnxslim==0.1.94

requirements: AutoUpdate success ✅ 3.2s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.21.0 

'/content/drive/MyDrive/RoadSense/runs/roadfix_yolo11s/weights/best.onnx'

## 🔄 Step 7 — Resume After Crash (from last checkpoint)

Run this if Colab disconnected mid-training. Automatically continues from the last saved checkpoint.

**Before running this cell, re-run Steps 1 → 5 first** (install, mount drive, config, download dataset).

In [ ]:
from ultralytics import YOLO
import os

LAST_PT = f'{DRIVE_PROJECT_PATH}/{RUN_NAME}/weights/last.pt'

if not os.path.exists(LAST_PT):
    print('❌ No checkpoint found at:', LAST_PT)
    print('   Run Step 6 (fresh training) instead.')
else:
    print(f'✅ Resuming from: {LAST_PT}')
    model = YOLO(LAST_PT)
    results = model.train(resume=True)
    print('✅ Training resumed and completed.')


## 🎯 Step 8 — Resume From a Specific Checkpoint Epoch

Use this if you want to restart from a specific saved epoch (e.g. epoch 50) rather than the very last one.

Change `RESUME_FROM_EPOCH` to the epoch number you want to load.

In [ ]:
from ultralytics import YOLO
import os

RESUME_FROM_EPOCH = 50   # ← change to whichever epoch you want

checkpoint = f'{DRIVE_PROJECT_PATH}/{RUN_NAME}/weights/epoch{RESUME_FROM_EPOCH}.pt'
weights_dir = f'{DRIVE_PROJECT_PATH}/{RUN_NAME}/weights/'

if not os.path.exists(checkpoint):
    print(f'❌ epoch{RESUME_FROM_EPOCH}.pt not found.')
    if os.path.exists(weights_dir):
        available = sorted([f for f in os.listdir(weights_dir) if f.startswith('epoch')])
        print(f'   Available checkpoints: {available}')
    else:
        print('   No checkpoints found. Run Step 6 first.')
else:
    print(f'✅ Loading checkpoint: epoch{RESUME_FROM_EPOCH}.pt')
    model = YOLO(checkpoint)
    results = model.train(resume=True)
    print(f'✅ Resumed from epoch {RESUME_FROM_EPOCH} and completed.')


## 📊 Step 9 — Validate Model

Evaluates the best saved model on the validation set and prints mAP, precision, and recall.

In [ ]:
from ultralytics import YOLO

BEST_PT = f'{DRIVE_PROJECT_PATH}/{RUN_NAME}/weights/best.pt'
model = YOLO(BEST_PT)

metrics = model.val(data=DATA_YAML)

print(f'\n📊 Validation Results:')
print(f'   mAP50:     {metrics.box.map50:.3f}')
print(f'   mAP50-95:  {metrics.box.map:.3f}')
print(f'   Precision: {metrics.box.mp:.3f}')
print(f'   Recall:    {metrics.box.mr:.3f}')


## 🖼️ Step 10 — Test on a Sample Image

Runs inference on a test image and displays the result with bounding boxes drawn.

By default uses the first image from the test set. Change `TEST_IMAGE` to any path or URL.

In [ ]:
from ultralytics import YOLO
from IPython.display import display, Image as IPImage
import glob, os

BEST_PT = f'{DRIVE_PROJECT_PATH}/{RUN_NAME}/weights/best.pt'
CONF_THRESHOLD = 0.4   # ← lower = more detections, higher = more confident only

# Auto-pick first test image from dataset
test_imgs = glob.glob('/content/roadfix-3/test/images/*.jpg')
TEST_IMAGE = test_imgs[0] if test_imgs else 'https://ultralytics.com/images/bus.jpg'
print(f'Testing on: {TEST_IMAGE}')

model = YOLO(BEST_PT)
results = model.predict(source=TEST_IMAGE, conf=CONF_THRESHOLD, save=True,
                        project='/content', name='test_output', exist_ok=True)

saved = glob.glob('/content/test_output/**/*.jpg', recursive=True)
if saved:
    display(IPImage(saved[0], width=700))
else:
    print('No output image found. Check the prediction ran correctly.')
